In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate", "-q"])

import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00
CUDA available: True
GPU count: 1
GPU 0: Tesla T4


In [3]:
DATASET_PATH = "/kaggle/input/datasets/sarveshwaran160405/failurecase/dataset_clean.jsonl"

pairs = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Total pairs: {len(pairs)}")

Total pairs: 7024


In [4]:
def format_pair(pair):
    return {"text": f"### Instruction:\n{pair['instruction']}\n\n### Response:\n{pair['response']}"}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

Train: 6672 | Test: 352


In [5]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda:0",
)

model = prepare_model_for_kbit_training(model)
print("Model loaded!")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [6]:
# Expanded target_modules to include the MLP/feed-forward layers
# (gate_proj, up_proj, down_proj), not just attention (q/k/v/o_proj).
# Attention-only LoRA mostly shifts response STYLE (tone, brevity, format).
# The MLP layers are where a transformer stores more of its factual/domain
# knowledge -- adapting them gives LoRA real capacity to shift *what the
# model knows*, not just how it phrases things. This directly targets the
# factual-accuracy gaps seen in the previous run (e.g. PVC binding causes,
# CPU throttling diagnostics) rather than just adjusting surface style.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [7]:
from transformers import EarlyStoppingCallback

# Previous run's own validation loss curve showed the best point around
# step ~1200 (~epoch 3 of 7), with eval loss steadily WORSENING for the
# remaining 4 epochs while train loss kept dropping -- textbook overfitting.
# Cutting to 4 epochs (with early stopping as a safety net) targets that
# same sweet spot directly, and should also roughly halve wall-clock time
# versus the previous ~5 hour run.
training_args = SFTConfig(
    output_dir="/kaggle/working/k8s-tinyllama",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training...")
trainer.train()
print("Training complete!")


Adding EOS to train dataset:   0%|          | 0/6672 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6672 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6672 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6672 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/352 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/352 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.256545,1.266490,1.299169,248749.000000,0.706968
400,1.243748,1.188359,1.215740,497001.000000,0.721381
600,1.049930,1.165743,1.115857,746276.000000,0.725598
800,1.045931,1.145404,1.123678,993798.000000,0.727659
1000,0.876794,1.171161,0.997913,1244515.000000,0.728307
1200,0.880840,1.152449,0.992625,1491502.000000,0.730901
1400,0.727699,1.205215,0.928982,1739297.000000,0.727405


Training complete!


In [8]:
OUTPUT_PATH = "/kaggle/working/k8s-tinyllama-final"
trainer.model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Saved to /kaggle/working/k8s-tinyllama-final


In [9]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    device="cuda:0",
)

test_queries = [
    "Why is my pod in CrashLoopBackOff?",
    "What causes OOMKilled errors?",
    "Why is my PVC stuck in Pending?",
    "Why is my deployment continuously restarting?",
    "How can I investigate CPU throttling?",
    "What are common causes of 5XX errors in Kubernetes?",
]

for query in test_queries:
    prompt = f"### Instruction:\n{query}\n\n### Response:\n"
    result = pipe(prompt, do_sample=False)
    print(f"Q: {query}")
    print(f"A: {result[0]['generated_text'].replace(prompt, '').strip()[:250]}")
    print()


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please 

Q: Why is my pod in CrashLoopBackOff?
A: Your pod is in CrashLoopBackOff because it is not able to reach the API server. The API server is not responding to requests from the pod, and the pod is stuck in a loop of restarting itself.



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What causes OOMKilled errors?
A: OOMKilled errors occur when the container's memory usage exceeds the container's memory limit, causing the container to be terminated.



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why is my PVC stuck in Pending?
A: Your PVC is stuck in Pending because the PersistentVolumeClaim (PVC) is not bound to a PersistentVolume (PV). You need to create a PV and then create a PVC to bind the PV to the PVC.



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why is my deployment continuously restarting?
A: Your deployment is continuously restarting because it is using a ConfigMap that is not yet available. The ConfigMap is not yet available because the Kubernetes API server is not yet ready.



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I investigate CPU throttling?
A: You can use the `top` command to monitor CPU usage, which will show the CPU usage of each process.

Q: What are common causes of 5XX errors in Kubernetes?
A: Common causes of 5XX errors include: incorrect API server configuration, incorrect API server configuration, incorrect API server configuration, incorrect API server configuration, incorrect API server configuration, incorrect API server configuratio

